In [ ]:
import numpy as np
import pandas as pd
import joblib
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# 1. Cargar el dataset final
df = pd.read_csv('dataset_final_colab.csv')

# 1. Asegurar que los datos base existan
# Reutilizamos el DataFrame y el escalador que ya tienes en memoria
try:
    # Verificamos si data_scaled ya existe, si no, hay que generarlo
    print(f"Verificando datos base... Dataset con {df.shape[1]} columnas.")
except NameError:
    print("⚠️ Error: No encontré el DataFrame 'df'. Asegúrate de haber cargado el CSV en la celda anterior.")

# 2. Definir las estaciones adicionales
# Ajusta estos nombres exactos a los de tu archivo CSV
estaciones_objetivo = ['ayacucho', 'caicara','ciudad_bolivar','palua']

# 3. Bucle de entrenamiento
for estacion in estaciones_objetivo:
    if estacion not in df.columns:
        print(f"❌ La estación '{estacion}' no existe en las columnas del dataset. Saltando...")
        continue

    print(f"\n" + "="*50)
    print(f"🛰️  ENTRENANDO ESPECIALISTA: {estacion.upper()}")
    print("="*50)

    # --- A. Creación de Ventanas para esta estación ---
    # Usamos la misma lógica de 60 días que en Ciudad Bolívar
    idx_target = df.columns.get_loc(estacion)

    X_esp = []
    y_esp = []

    # Re-creamos las ventanas para asegurar consistencia
    for i in range(60, len(data_scaled)):
        X_esp.append(data_scaled[i-60:i, :]) # Las 27 variables de entrada
        y_esp.append(data_scaled[i, idx_target]) # El nivel de la estación actual

    X_esp = np.array(X_esp)
    y_esp = np.array(y_esp)

    # Split 80/20 (Temporal)
    split = int(len(X_esp) * 0.8)
    X_train, X_test = X_esp[:split], X_esp[split:]
    y_train, y_test = y_esp[:split], y_esp[split:]

    # --- B. Arquitectura del Modelo (Misma base que el primero) ---
    model = Sequential([
        LSTM(100, return_sequences=True, input_shape=(60, X_esp.shape[2])),
        Dropout(0.2),
        LSTM(50, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])

    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    # --- C. Callbacks ---
    # EarlyStopping para evitar sobreentrenamiento
    monitor = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5)

    # --- D. Ejecución ---
    print(f"Entrenando {estacion}...")
    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_test, y_test),
        callbacks=[monitor, reduce_lr],
        verbose=1 # Cambiado a 1 para que veas el progreso de cada una
    )

    # --- E. Guardar Resultados ---
    model.save(f'modelo_{estacion}.h5')
    print(f"✅ Modelo guardado como: modelo_{estacion}.h5")

print("\n" + "="*50)
print("🎯 ¡PROCESO FINALIZADO CON ÉXITO!")
print("Ya tienes un modelo especialista para cada punto del río.")

In [ ]:
#escaladores 
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib

# 1. Cargar tu dataset
df = pd.read_csv('dataset_final_colab.csv')

# 2. Filtrar solo columnas numéricas (quitando 'fecha' si existe)
columnas_numericas = df.select_dtypes(include=[np.number]).columns
df_numeric = df[columnas_numericas]

# --- ESCALADOR GLOBAL (Para las entradas X) ---
# Este se usa para transformar las 27 variables antes de meterlas al modelo
scaler_x = MinMaxScaler(feature_range=(0, 1))
scaler_x.fit(df_numeric)
joblib.dump(scaler_x, 'scaler_x_final.pkl')
print("✅ scaler_x_final.pkl creado (27 variables).")

# --- ESCALADORES INDIVIDUALES (Para las salidas Y) ---
# Creamos uno por estación para poder convertir la predicción a metros fácilmente
estaciones = ['ayacucho', 'caicara', 'ciudad_bolivar', 'palua']

for est in estaciones:
    if est in df.columns:
        scaler_y = MinMaxScaler(feature_range=(0, 1))
        # Ajustamos el escalador solo con los datos de esa columna
        scaler_y.fit(df[[est]]) 
        
        nombre_archivo = f'scaler_y_{est}.pkl'
        joblib.dump(scaler_y, nombre_archivo)
        print(f"✅ {nombre_archivo} creado.")

print("\n--- ¡LISTO! ---")
print("Descarga todos los archivos .pkl de la carpeta de la izquierda.")